# CSE 151B — Submission notebook (`private.jsonl`)

Live in **`notebooks/submission.ipynb`**. Paths to `data/` and `results/` resolve automatically from **repo root**, whether the Jupyter cwd is the project root or `notebooks/`.

Runs inference on **every row** of `data/private.jsonl` (no ground-truth labels) and writes the competition **CSV** upload:

- **Columns:** `id`, `response` (full model trace, properly CSV-escaped)
- **Checkpoint:** `results/submission.responses.jsonl` — append-only resume if the run stops mid-way

Pipeline matches **`notebooks/dev.ipynb`** (prompts, vLLM INT8 load, `MAX_TOKENS=8192`), but generation uses **MCQ vs free-form `SamplingParams`** in separate batches per chunk.

**Colab:** GPU runtime → `%pip` cell → restart → optional Drive copy for `private.jsonl` → run all (run **Imports & configuration** first so `REPO_ROOT` exists).

## 1. Environment (same stack as dev / starter)

**Colab:** run below, **Runtime → Restart runtime**, continue. **Local:** use your project venv with `vllm`, `transformers`, `bitsandbytes`, etc.

In [ ]:
# Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers "bitsandbytes>=0.48.1"

## 2. Imports & configuration

In [ ]:
import csv
import json
import os

from pathlib import Path


def _repo_root() -> Path:
    """Project root (parent of `data/`), whether cwd is repo root or `notebooks/`."""
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()
MODEL_ID       = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID         = "0"
DATA_PATH      = str(REPO_ROOT / "data/private.jsonl")
SUBMISSION_CSV = str(REPO_ROOT / "results/submission.csv")
MAX_TOKENS     = 8192
CHUNK_SIZE     = 100

print(f"REPO_ROOT={REPO_ROOT}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

## 3. Colab: copy `private.jsonl` from Drive

Upload **`private.jsonl`** to `My Drive/CSE151B/data/private.jsonl` (or change `DRIVE_PRIVATE`). Skip on local clone when `data/private.jsonl` exists.

In [ ]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use repo `data/private.jsonl`.")
else:
    drive.mount("/content/drive")
    DRIVE_PRIVATE = Path("/content/drive/MyDrive/CSE151B/data/private.jsonl")
    DRIVE_PROJECT_ROOT = DRIVE_PRIVATE.parent.parent
    if not DRIVE_PRIVATE.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_PRIVATE}\nUpload private.jsonl or set DRIVE_PRIVATE."
        )
    (REPO_ROOT / "data").mkdir(parents=True, exist_ok=True)
    dest = REPO_ROOT / "data/private.jsonl"
    shutil.copy2(DRIVE_PRIVATE, dest)
    print(f"Copied to {dest.resolve()}")

## 4. Load dataset

Private rows have `question`, optional `options`, and `id` — **no** `answer` field.

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]
n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = len(data) - n_mcq
print(f"Loaded {len(data)} problems from {DATA_PATH}  ({n_mcq} MCQ, {n_free} free-form)")

## 5. Prompt construction (same as `notebooks/dev.ipynb`)

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


def chat_prompt_text(tokenizer, question: str, options: Optional[list]) -> str:
    system, user = build_prompt(question, options)
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )

## 6. Load model (vLLM + bitsandbytes, same as dev)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        quantization="bitsandbytes",
        load_format="bitsandbytes",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.88,
        max_model_len=16384,
        trust_remote_code=True,
        max_num_seqs=64,
        max_num_batched_tokens=16384,
    )

sampling_params_free = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

sampling_params_mcq = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.2,
    top_p=0.9,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

## 7. Generate all responses (checkpointed)

Deletes **`submission.responses.jsonl`** only if you want a full rerun from scratch.

In [ ]:
try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(SUBMISSION_CSV).parent

_results_dir.mkdir(parents=True, exist_ok=True)
response_checkpoint = _results_dir / (Path(SUBMISSION_CSV).stem + ".responses.jsonl")
print(f"Checkpoint path: {response_checkpoint}")

completed: dict = {}
if response_checkpoint.exists():
    with open(response_checkpoint) as f:
        for line in f:
            r = json.loads(line)
            completed[r["id"]] = r["response"]
    print(f"Resumed from checkpoint: {len(completed)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed]
print(f"Generating {len(remaining)} remaining / {len(data)} total...")


def append_checkpoint(rows: list[tuple[int, str]]) -> None:
    with open(response_checkpoint, "a") as f:
        for qid, response in rows:
            f.write(json.dumps({"id": qid, "response": response}) + "\n")


def generate_batch(items: list, sampling_params: SamplingParams) -> None:
    if not items:
        return
    prompts = [
        chat_prompt_text(tokenizer, it["question"], it.get("options"))
        for it in items
    ]
    with _jupyter_stdout_for_vllm():
        outputs = llm.generate(prompts, sampling_params=sampling_params)
    batch_lines: list[tuple[int, str]] = []
    for item, out in zip(items, outputs):
        response = out.outputs[0].text.strip()
        completed[item["id"]] = response
        batch_lines.append((item["id"], response))
    append_checkpoint(batch_lines)


_chunk_iter = range(0, len(remaining), CHUNK_SIZE)
if tqdm is not None:
    _chunk_iter = tqdm(
        _chunk_iter,
        total=(len(remaining) + CHUNK_SIZE - 1) // CHUNK_SIZE,
        desc="Chunks",
    )

for chunk_start in _chunk_iter:
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]
    mcq_chunk  = [x for x in chunk if x.get("options")]
    free_chunk = [x for x in chunk if not x.get("options")]

    generate_batch(mcq_chunk, sampling_params_mcq)
    generate_batch(free_chunk, sampling_params_free)

    done = len(completed)
    print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

assert len(completed) == len(data), "Missing ids — checkpoint vs DATA_PATH mismatch"
responses = [completed[d["id"]] for d in data]
print(f"Done. {len(responses)} responses ready.")

## 8. Write submission CSV

Uses Python’s `csv` writer so commas and newlines inside `response` are quoted per RFC 4180.

In [ ]:
try:
    csv_out = DRIVE_PROJECT_ROOT / "results" / Path(SUBMISSION_CSV).name
except NameError:
    csv_out = Path(SUBMISSION_CSV)

csv_out.parent.mkdir(parents=True, exist_ok=True)

with open(csv_out, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
    w.writerow(["id", "response"])
    for row in data:
        qid = row["id"]
        w.writerow([qid, completed[qid]])

print(f"Wrote {len(data)} rows to {csv_out.resolve()}")

with open(csv_out, newline="", encoding="utf-8") as f:
    n = sum(1 for _ in f)
assert n == len(data) + 1, f"Expected header + {len(data)} rows, got {n} lines"
print("CSV line count OK (1 header +", len(data), "data rows).")

## Done

Upload **`submission.csv`** to the course leaderboard workflow. Keep **`submission.responses.jsonl`** as a backup of raw traces.